In [ ]:
from headlines.bert.data import load_csv

ds = load_csv("data/guardian_headlines.csv")
ds

In [5]:
from headlines.bert.config import BertCFG

model_name = BertCFG.model_name

In [7]:
splits = ds.train_test_split(test_size=0.2, seed=42)
test_val = splits["test"].train_test_split(test_size=0.5, seed=42)

train = splits["train"]
val = test_val["train"]
test = test_val["test"]

In [12]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")



def tokenize(example):
    """tokenize data for training"""
    return tokenizer(
        example['Headlines'],
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )
    
all_columns = ds.select_columns
train_tokenized = train.map(tokenize, batched=True, remove_columns=['Headlines'])
test_tokenized = test.map(tokenize, batched=True, remove_columns=['Headlines'])
val_tokenized = val.map(tokenize, batched=True, remove_columns=['Headlines'])  

Map: 100%|██████████| 1780/1780 [00:00<00:00, 7824.12 examples/s]


In [13]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def eval(eval_pred):
    """evaluate results"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return {
        "accuracy": accuracy_score(predictions, labels),
        "f1 score": f1_score(predictions, labels, average="weighted")
    }
    



In [ ]:
from transformers import TrainingArguments, Trainer

training_arguments = TrainingArguments(
    output_dir=